# Practical 12: Quadratic Forms & Application using SageMath

## 1. Symmetric matrix of Q(x, y) = 2x² + 6xy + 5y²
Diagonal entries are the coefficients of x², y²; each off-diagonal entry is half the xy coefficient (6/2 = 3).

In [ ]:
var('x y')
A = matrix(QQ, [[2, 3],
                [3, 5]])
X = vector([x, y])
Q = (X * A * X).expand()   # x^T A x
print("A =")
print(A)
print("x^T A x =", Q)
print("Matches Q(x,y)?", bool(Q == 2*x^2 + 6*x*y + 5*y^2))

*Conclusion: xᵀAx reproduces Q(x, y) exactly.*

## 2. Classification using eigenvalues

In [ ]:
def classify(M):
    ev = M.eigenvalues()
    if all(e > 0 for e in ev): t = "Positive definite"
    elif all(e < 0 for e in ev): t = "Negative definite"
    elif all(e >= 0 for e in ev): t = "Positive semi-definite"
    elif all(e <= 0 for e in ev): t = "Negative semi-definite"
    else: t = "Indefinite"
    return [RR(e).n(digits=5) for e in ev], t

mats = {
    "(a)": matrix(QQ, [[5, 0], [0, 2]]),
    "(b)": matrix(QQ, [[1, 1], [1, 1]]),
    "(c)": matrix(QQ, [[1, 0], [0, -1]]),
    "(d)": matrix(QQ, [[3, 1, 0], [1, 2, 0], [0, 0, 1]])
}
for name, M in mats.items():
    ev, t = classify(M)
    print(name, "eigenvalues:", ev, "->", t)

*Conclusion: (a) PD, (b) PSD, (c) Indefinite, (d) PD. For (d) the exact eigenvalues are 1 and (5 ± √5)/2. (Order of eigenvalues may vary.)*

## 3. Principal axes form of Q(x, y) = 3x² + 2xy + 3y²
The symmetric matrix is A = [[3, 1], [1, 3]].

In [ ]:
A = matrix(QQ, [[3, 1],
                [1, 3]])
print("Eigenvalues:", A.eigenvalues())
cols = []; lams = []
for lam, vecs, mult in A.eigenvectors_right():
    v = vecs[0]
    u = v / v.norm()          # normalise -> orthonormal
    lams.append(lam); cols.append(u)
    print("lambda =", lam, " unit eigenvector =", u)
P = matrix(cols).transpose()  # orthogonal matrix
print("P ="); print(P)
print("P^T P ="); print((P.transpose()*P).simplify_full())
print("P^T A P ="); print((P.transpose()*A*P).simplify_full())
var('u v')
print("Principal axes form: Q =", lams[0]*u^2 + lams[1]*v^2)

*Conclusion: With x = Pu (u = (x + y)/√2, v = (x − y)/√2), Q = 4u² + 2v², which is positive definite.*

## 4. SVD of A = [[3, 0], [4, 5]]

In [ ]:
A = matrix(RDF, [[3, 0],
                 [4, 5]])
# (a) SVD
U, S, V = A.SVD()
print("U ="); print(U)
print("S ="); print(S)
print("V ="); print(V)
# (b) Verify A = U S V^T
print("U*S*V^T ="); print((U*S*V.transpose()).round(10))
print("Error ||A - U S V^T|| =", (A - U*S*V.transpose()).norm())
# (c) Singular values vs sqrt(eigenvalues of A^T A)
sv = S.diagonal()
ev = (A.transpose()*A).eigenvalues()
print("A^T A ="); print(A.transpose()*A)
print("Singular values from S :", sv)
print("Eigenvalues of A^T A :", sorted(ev, reverse=True))
print("Square roots :", [sqrt(e) for e in sorted(ev, reverse=True)])

*Conclusion: The singular values 6.7082 = 3√5 and 2.2361 = √5 are exactly the square roots of the eigenvalues 45 and 5 of AᵀA, and U S Vᵀ = A up to rounding.*

## 5. SVD of B = [[1, 1], [0, 0]] and its rank

In [ ]:
B = matrix(RDF, [[1, 1],
                 [0, 0]])
U, S, V = B.SVD()
print("U ="); print(U)
print("S ="); print(S)
print("V ="); print(V)
print("Check U*S*V^T ="); print((U*S*V.transpose()).round(10))
sv = S.diagonal()
r = len([s for s in sv if abs(s) > 1e-10])   # count non-zero singular values
print("Singular values:", sv)
print("Rank from singular values =", r)
print("B.rank() =", B.rank())

*Conclusion: B has one non-zero singular value (√2) and one zero singular value, so rank(B) = 1. The rank of a matrix equals the number of its non-zero singular values.*

## 6. PageRank of a 3-page web
P1→ P2, P3; P2→ P3; P3→ P1. Column j holds the outgoing links of page Pj, each with weight 1/(number of outlinks).

In [ ]:
# (a) Column-stochastic link matrix
L = matrix(QQ, [[0,   0, 1],
                [1/2, 0, 0],
                [1/2, 1, 0]])
print("L ="); print(L)
print("Column sums:", [sum(c) for c in L.columns()])
# (b) 1 is an eigenvalue
print("Characteristic polynomial:", L.charpoly())
print("Eigenvalues:", L.eigenvalues())
print("det(L - I) =", (L - identity_matrix(3)).det())
v = (L - identity_matrix(3)).right_kernel().basis()[0]
print("Eigenvector for lambda = 1:", v)
print("Check L*v == v:", L*v == v)
# (c) Normalise so entries sum to 1, then rank
p = v / sum(v)
print("PageRank vector:", p, "=", p.n(digits=4))
rank = sorted(range(3), key=lambda i: -p[i])
print("Ranking:", ["P%d (%s)" % (i+1, p[i]) for i in rank])

*Conclusion: PageRank = (0.4, 0.2, 0.4). P1 and P3 are tied as most important; P2 is least important.*

## 7. Power iteration with x0 = (1/3, 1/3, 1/3)

In [ ]:
L = matrix(RDF, [[0,   0, 1],
                 [1/2, 0, 0],
                 [1/2, 1, 0]])
x0 = vector(RDF, [1/3, 1/3, 1/3])
# (a) x_k = L^k x0
for k in [5, 10, 20]:
    xk = (L^k) * x0
    print("k =", k, ": x_k =", xk.n(digits=6))
# (b) Compare with the PageRank vector from Problem 6
p = vector(RDF, [2/5, 1/5, 2/5])
print("PageRank vector p =", p)
print("||x_20 - p|| =", ((L^20)*x0 - p).norm())

*Conclusion: x_k approaches (0.4, 0.2, 0.4), the PageRank vector found in Problem 6. It converges because the other eigenvalues of L have modulus 1/√2 < 1.*

## 8. Damped PageRank (Google matrix), d = 0.85, n = 4
P1→ P2, P3; P2→ P3, P4; P3→ P1; P4→ P1, P3. G = d·L + ((1 − d)/n)·J.

In [ ]:
# (a) Build G over RDF
L = matrix(RDF, [[0,   0,   1, 1/2],
                 [1/2, 0,   0, 0  ],
                 [1/2, 1/2, 0, 1/2],
                 [0,   1/2, 0, 0  ]])
d = 0.85; n = 4
J = matrix(RDF, n, n, [1]*(n*n))   # all-ones matrix
G = d*L + ((1 - d)/n)*J
print("G ="); print(G)
print("Column sums:", [sum(c) for c in G.columns()])

# (b) Power iteration for the dominant eigenvector
x = vector(RDF, [1/n]*n)
for k in range(1, 201):
    x_new = G*x
    x_new = x_new / sum(x_new)
    if (x_new - x).norm() < 1e-12:
        x = x_new; break
    x = x_new
print("Converged after", k, "iterations")
print("PageRank vector:", x.n(digits=5))
print("Check G*x == x:", (G*x - x).norm() < 1e-10)

# (c) Rank the pages
order = sorted(range(n), key=lambda i: -x[i])
for pos, i in enumerate(order, 1):
    print(pos, ": P%d score = %.4f" % (i+1, x[i]))

*Conclusion: Ranking: P1 (0.3640) > P3 (0.3246) > P2 (0.1922) > P4 (0.1192).*